# CogMem BigCodeBench Episode Collection

Collect Q-valued episodes from BigCodeBench using Qwen2.5:3b.

**Dataset subsets:**
- **Full** -- 1140 tasks (`USE_HARD = False`)
- **Hard** -- 148 tasks (`USE_HARD = True`)

Set `USE_HARD` in **Cell 6** before running the main collection.

**Flow:** Run cells 1-5 (setup), configure Cell 6, then run 6-10.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [1]:
# Cell 2: Install system deps + Python packages + Ollama
!apt-get update -qq && apt-get install -y -qq build-essential cmake zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q transformers peft accelerate datasets huggingface-hub pyyaml openai trl

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%############                                                  33.4%
>>> Creating ollama user...
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 2.2.2 requires transformers<5.0.0,>=4.6.0, but you have tr

In [2]:
# Cell 3: Start Ollama + pull model
import subprocess, time, os

proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

!ollama pull qwen2.5:3b
print("Ollama + Qwen2.5:3b ready!")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 5ee4f07cdb9b:   2% ▕                  ▏  29 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   6% ▕█                 ▏ 120 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  12% ▕██                ▏ 225 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  14% ▕██                ▏ 267 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  19% ▕███               ▏ 371 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  24% ▕████              ▏ 458 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  26% ▕████              ▏ 503 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  31% ▕█████             ▏ 602 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  38% ▕██████            ▏ 725 MB/1.9 GB    

In [3]:
# Cell 4: Clone CogMem + load BigCodeBench datasets
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || (cd /notebooks/CogMem && git pull && git checkout feat/bigcodebench-integration)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys, json
from pathlib import Path
from datasets import load_dataset

if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import cogmem
print(f"cogmem loaded from {cogmem.__file__}")

def load_and_save(dataset_name, split, save_path):
    if Path(save_path).exists():
        with open(save_path) as f:
            tasks = [json.loads(line) for line in f if line.strip()]
        print(f"Loaded from cache: {len(tasks)} tasks ({save_path})")
        return tasks
    ds = load_dataset(dataset_name, split=split)
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "canonical_solution": item.get("canonical_solution", ""),
            "entry_point": item.get("entry_point", ""),
        })
    with open(save_path, "w") as f:
        for t in tasks:
            f.write(json.dumps(t) + "\n")
    print(f"Downloaded: {len(tasks)} tasks -> {save_path}")
    return tasks

full_tasks = load_and_save("bigcode/bigcodebench", "v0.1.4", "/notebooks/bigcodebench_tasks.jsonl")
hard_tasks = load_and_save("bigcode/bigcodebench-hard", "v0.1.4", "/notebooks/bigcodebench_hard_tasks.jsonl")

print(f"\nReady: Full={len(full_tasks)}, Hard={len(hard_tasks)}")

remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 31 (delta 19), reused 29 (delta 17), pack-reused 0 (from 0)
Unpacking objects: 100% (31/31), 27.16 KiB | 132.00 KiB/s, done.
From https://github.com/tungooxx/CogMem
   a3bc8df..79c17f0  feat/bigcodebench-integration -> origin/feat/bigcodebench-integration
Updating a3bc8df..79c17f0
Fast-forward
 cogmem/config.py                        | 200 ++++++------
 cogmem/consolidation/__init__.py        |   5 +-
 cogmem/consolidation/abstract.py        | 138 ++++++++-
 cogmem/consolidation/pipeline.py        | 290 ++++++++---------
 cogmem/consolidation/train_generator.py | 319 +++++++++++++++++++
 cogmem/consolidation/train_verifier.py  | 134 ++++++++
 cogmem/inference/__init__.py            |   1 +
 cogmem/inference/verify_candidates.py   |  80 +++++
 paperspace_bigcode.ipynb                | 531 +++++++++++++-------------------
 paperspace_bigcode_

In [4]:
# Cell 5: Sanity check -- run 3 tasks with qwen2.5:3b
from openai import OpenAI
from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

for task in full_tasks[:3]:
    messages = format_messages(task, use_instruct=True)
    resp = client.chat.completions.create(
        model="qwen2.5:3b", messages=messages,
        max_tokens=2048, temperature=0,
    )
    response = resp.choices[0].message.content
    code = extract_code(response)
    result = evaluate_solution(task, code, timeout=30, mode="subprocess")
    status = "PASS" if result["passed"] else "FAIL"
    print(f"{task['task_id']}: {status}")
    if not result["passed"] and result.get("error"):
        print(f"  Error: {result['error'][:200]}")
    print()

print("Sanity check done!")

BigCodeBench/0: PASS

BigCodeBench/1: PASS

BigCodeBench/2: FAIL
  Error: ======================================================================
FAIL: test_case_2 (__main__.TestCases.test_case_2)
----------------------------------------------------------------------
Traceba

Sanity check done!


In [5]:
# Cell 6: Main collection -- configure USE_HARD here
# ============================================================
MODEL = "qwen2.5:3b"
USE_HARD = False  # True for Hard-148, False for Full-1140
# ============================================================

import json, time, sys
from pathlib import Path
from openai import OpenAI
from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

model_tag = MODEL.replace(":", "_").replace(".", "_")
subset_tag = "hard" if USE_HARD else "full"
CHECKPOINT = f"/notebooks/bigcode_{model_tag}_{subset_tag}.jsonl"
TASKS_PATH = "/notebooks/bigcodebench_hard_tasks.jsonl" if USE_HARD else "/notebooks/bigcodebench_tasks.jsonl"

print(f"Model:      {MODEL}")
print(f"Subset:     {subset_tag}")
print(f"Checkpoint: {CHECKPOINT}")

# Load tasks
tasks = []
with open(TASKS_PATH) as f:
    for line in f:
        tasks.append(json.loads(line.strip()))
print(f"Tasks: {len(tasks)}")

# Resume
completed_ids = set()
episodes = []
if Path(CHECKPOINT).exists():
    with open(CHECKPOINT, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                ep = json.loads(line)
                episodes.append(ep)
                completed_ids.add(ep["task_id"])
            except json.JSONDecodeError:
                break
    print(f"Resumed: {len(completed_ids)} done")

remaining = [t for t in tasks if t["task_id"] not in completed_ids]
total = len(tasks)
done = len(completed_ids)
passed = sum(1 for ep in episodes if ep["success"])

if not remaining:
    print(f"\nAll {total} tasks done! {passed}/{total} ({passed/total:.1%})")
else:
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    start_time = time.time()

    print(f"Remaining: {len(remaining)}")
    print("=" * 60)

    for i, task in enumerate(remaining):
        try:
            messages = format_messages(task, use_instruct=True)
            resp = client.chat.completions.create(
                model=MODEL, messages=messages,
                max_tokens=2048, temperature=0,
            )
            response = resp.choices[0].message.content
            code = extract_code(response)
            result = evaluate_solution(task, code, timeout=30, mode="subprocess")

            episode = {
                "episode_id": f"bigcode_{task['task_id'].replace('/', '_')}_{int(time.time())}",
                "task_id": task["task_id"],
                "task_type": f"bigcodebench_{subset_tag}",
                "task_description": task.get("instruct_prompt", task.get("complete_prompt", "")),
                "script": response,
                "generated_code": code,
                "success": result["passed"],
                "q_value": 1.0 if result["passed"] else -1.0,
                "error": result.get("error"),
                "entry_point": task.get("entry_point", ""),
                "model": MODEL,
                "timestamp": time.time(),
            }
        except Exception as e:
            episode = {
                "episode_id": f"bigcode_{task['task_id'].replace('/', '_')}_{int(time.time())}",
                "task_id": task["task_id"],
                "task_type": f"bigcodebench_{subset_tag}",
                "task_description": task.get("instruct_prompt", ""),
                "script": "", "generated_code": "",
                "success": False, "q_value": -1.0,
                "error": str(e),
                "entry_point": task.get("entry_point", ""),
                "model": MODEL,
                "timestamp": time.time(),
            }

        episodes.append(episode)
        done += 1
        if episode["success"]:
            passed += 1

        with open(CHECKPOINT, "a", encoding="utf-8") as f:
            f.write(json.dumps(episode, ensure_ascii=False) + "\n")

        if (i + 1) % 10 == 0 or i < 5:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
            eta = (len(remaining) - i - 1) / rate * 3600 if rate > 0 else 0
            status = "PASS" if episode["success"] else "FAIL"
            print(f"[{done}/{total}] {task['task_id']}: {status} | "
                  f"Pass: {passed}/{done} ({passed/done:.1%}) | "
                  f"Rate: {rate:.0f}/hr | ETA: {eta/60:.0f}m")

    print("=" * 60)
    print(f"DONE! {MODEL}: {passed}/{done} ({passed/done:.1%})")
    print(f"Checkpoint: {CHECKPOINT}")

Model:      qwen2.5:3b
Subset:     full
Checkpoint: /notebooks/bigcode_full_qwen2.5_3b.jsonl
Tasks: 1140
Remaining: 1140
[1/1140] BigCodeBench/0: PASS | Pass: 1/1 (100.0%) | Rate: 279/hr | ETA: 245m
[2/1140] BigCodeBench/1: PASS | Pass: 2/2 (100.0%) | Rate: 495/hr | ETA: 138m
[3/1140] BigCodeBench/2: FAIL | Pass: 2/3 (66.7%) | Rate: 642/hr | ETA: 106m
[4/1140] BigCodeBench/3: PASS | Pass: 3/4 (75.0%) | Rate: 766/hr | ETA: 89m
[5/1140] BigCodeBench/4: PASS | Pass: 4/5 (80.0%) | Rate: 895/hr | ETA: 76m
[10/1140] BigCodeBench/9: PASS | Pass: 6/10 (60.0%) | Rate: 1085/hr | ETA: 63m
[20/1140] BigCodeBench/19: FAIL | Pass: 10/20 (50.0%) | Rate: 769/hr | ETA: 87m
[30/1140] BigCodeBench/29: FAIL | Pass: 15/30 (50.0%) | Rate: 901/hr | ETA: 74m
[40/1140] BigCodeBench/39: FAIL | Pass: 16/40 (40.0%) | Rate: 919/hr | ETA: 72m
[50/1140] BigCodeBench/49: FAIL | Pass: 17/50 (34.0%) | Rate: 923/hr | ETA: 71m
[60/1140] BigCodeBench/59: FAIL | Pass: 19/60 (31.7%) | Rate: 944/hr | ETA: 69m
[70/1140] BigCo

In [6]:
# Cell 7: Q-value analysis
import json
from collections import Counter
from pathlib import Path

# Re-derive CHECKPOINT from Cell 6 variables
model_tag = MODEL.replace(":", "_").replace(".", "_")
subset_tag = "hard" if USE_HARD else "full"
CHECKPOINT = f"/notebooks/bigcode_{model_tag}_{subset_tag}.jsonl"

# Load episodes
episodes = []
with open(CHECKPOINT, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                episodes.append(json.loads(line))
            except json.JSONDecodeError:
                break

total = len(episodes)
passed = [ep for ep in episodes if ep["success"]]
failed = [ep for ep in episodes if not ep["success"]]

print(f"=== Q-Value Analysis: {MODEL} ({subset_tag}) ===")
print(f"Total:  {total}")
print(f"Passed: {len(passed)} ({len(passed)/total:.1%})")
print(f"Failed: {len(failed)} ({len(failed)/total:.1%})")

# Q-value zones
q_values = [ep["q_value"] for ep in episodes]
high_q = sum(1 for q in q_values if q > 0.5)
mid_q = sum(1 for q in q_values if -0.5 <= q <= 0.5)
low_q = sum(1 for q in q_values if q < -0.5)
print(f"\n--- Q-Value Zones ---")
print(f"High (q > 0.5):   {high_q}")
print(f"Mid  (-0.5..0.5): {mid_q}")
print(f"Low  (q < -0.5):  {low_q}")

# Error breakdown
errors = []
for ep in failed:
    err = ep.get("error", "") or ""
    if "Timeout" in err:
        errors.append("Timeout")
    elif "SyntaxError" in err:
        errors.append("SyntaxError")
    elif "ImportError" in err or "ModuleNotFoundError" in err:
        errors.append("ImportError")
    elif "NameError" in err:
        errors.append("NameError")
    elif "TypeError" in err:
        errors.append("TypeError")
    elif "AttributeError" in err:
        errors.append("AttributeError")
    elif "AssertionError" in err or "TEST_FAILED" in err:
        errors.append("TestFailure")
    elif err:
        errors.append("Other")
    else:
        errors.append("Unknown")

print(f"\n--- Error Breakdown ---")
for err_type, count in Counter(errors).most_common():
    print(f"  {err_type:20s} {count:4d} ({count/len(failed):.1%})")

# Passed tasks list
print(f"\n--- Passed Tasks ({len(passed)}) ---")
for ep in sorted(passed, key=lambda x: x["task_id"]):
    print(f"  {ep['task_id']}")

# Sample failures
print(f"\n--- Sample Failures (first 5) ---")
for ep in failed[:5]:
    err_preview = (ep.get("error") or "no error message")[:150]
    print(f"  {ep['task_id']}: {err_preview}")

FileNotFoundError: [Errno 2] No such file or directory: '/notebooks/bigcode_qwen2_5_3b_full.jsonl'

In [ ]:
# Cell 8: Show results for all models
import glob, json

checkpoint_files = sorted(glob.glob("/notebooks/bigcode_*.jsonl"))

if not checkpoint_files:
    print("No checkpoint files found yet.")
else:
    print(f"{'File':<55s} {'Total':>6s} {'Pass':>6s} {'Rate':>8s}")
    print("-" * 77)
    for fpath in checkpoint_files:
        eps = []
        with open(fpath, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        eps.append(json.loads(line))
                    except json.JSONDecodeError:
                        break
        total = len(eps)
        passed = sum(1 for ep in eps if ep["success"])
        rate = passed / total if total > 0 else 0.0
        fname = fpath.split("/")[-1]
        print(f"{fname:<55s} {total:>6d} {passed:>6d} {rate:>7.1%}")

In [7]:
# Cell 9: Save as memory bank JSON
import json, sys
from pathlib import Path

if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

# Re-derive paths from Cell 6 variables
model_tag = MODEL.replace(":", "_").replace(".", "_")
subset_tag = "hard" if USE_HARD else "full"
CHECKPOINT = f"/notebooks/bigcode_{model_tag}_{subset_tag}.jsonl"
model_name = f"{model_tag}_{subset_tag}"
bank_path = f"/notebooks/CogMem/results/memory_bank_{model_name}.json"

# Load episodes from checkpoint
episodes = []
with open(CHECKPOINT, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                episodes.append(json.loads(line))
            except json.JSONDecodeError:
                break

# Save as memory bank
Path(bank_path).parent.mkdir(parents=True, exist_ok=True)
with open(bank_path, "w", encoding="utf-8") as f:
    json.dump(episodes, f, indent=2, ensure_ascii=False)

total = len(episodes)
passed = sum(1 for ep in episodes if ep["success"])
rate = passed / total if total > 0 else 0.0

print(f"Memory bank saved: {bank_path}")
print(f"Episodes: {total}, Passed: {passed} ({rate:.1%})")
print(f"File size: {Path(bank_path).stat().st_size / 1024 / 1024:.1f} MB")

FileNotFoundError: [Errno 2] No such file or directory: '/notebooks/bigcode_qwen2_5_3b_full.jsonl'

In [ ]:
# Cell 10: Memory bank table -- every episode with task description
import json

# Re-derive paths from Cell 6 variables
model_tag = MODEL.replace(":", "_").replace(".", "_")
subset_tag = "hard" if USE_HARD else "full"
CHECKPOINT = f"/notebooks/bigcode_{model_tag}_{subset_tag}.jsonl"

episodes = []
with open(CHECKPOINT, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            try:
                episodes.append(json.loads(line))
            except json.JSONDecodeError:
                break

def classify_error(err):
    if not err: return "-"
    if "Timeout" in err: return "Timeout"
    if "SyntaxError" in err: return "Syntax"
    if "ImportError" in err or "ModuleNotFoundError" in err: return "Import"
    if "NameError" in err: return "Name"
    if "TypeError" in err: return "Type"
    if "TEST_FAILED" in err or "AssertionError" in err: return "TestFail"
    return "Other"

def short_desc(ep):
    desc = ep.get("task_description", "")
    first_line = desc.split("
")[0].strip()
    if len(first_line) > 40:
        return first_line[:37] + "..."
    return first_line

print(f"{'#':>4}  {'Task ID':<22} {'Q':>5}  {'ok':>2}  {'Description':<42} {'Error':<10}")
print("-" * 90)

for i, ep in enumerate(episodes, 1):
    tid = ep["task_id"]
    q = ep["q_value"]
    ok = "Y" if ep["success"] else "N"
    desc = short_desc(ep)
    err = classify_error(ep.get("error", ""))
    print(f"{i:>4}  {tid:<22} {q:>5.1f}  {ok:>2}  {desc:<42} {err:<10}")

total = len(episodes)
passed = sum(1 for ep in episodes if ep["success"])
print(f"
Total: {total} | Passed: {passed} ({passed/total:.1%})")
